### Generating Synthetic Data and Performing DUC

In [ ]:
import numpy as np
from DeLUCA import DeLUCA 
from custom_funcs import missing_data_generation, thrC, post_proC, err_rate, generate_data
from dataset_params import Dataset_params
from dataset_params import create_log
import torch
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=UserWarning)

def run_model(missing_data_perc, custom_model, m, n, d, K, noise):
    torch.manual_seed(17)
    data, full_data, input_shape, batch_size, total_datapoints, \
    flat_layer_size, enc_layer_size, deco_layer_size, kernel_size, output_padding, \
    K, reg1, reg2, alpha1, alpha2, d, lr, rank, \
    true_labels = generate_data(m, n, d, K, noise)  
    
    logs_path = create_log("synthetic_data_"+str(K)+"_"+str(d))
    cluster_model = custom_model  
    missing_data_input = missing_data_generation(data, int(missing_data_perc * total_datapoints))

    CAE = DeLUCA(input_shape, flat_layer_size, enc_layer_size, deco_layer_size, 
                kernel_size, output_padding, lr, K, rank, reg_const1=reg1, reg_const2=reg2,
                batch_size=batch_size, model_path=None, logs_path=logs_path,
                cluster_model=cluster_model, device=device)
    CAE.to(device)
    stopping_lr = lr/10
    data_norm = np.linalg.norm(full_data)
    iter = 1
    accuracy = 0
    while iter:
        C, cost, complete_data, lr = CAE.finetune_fit(missing_data_input)
        accuracy = 1 - np.linalg.norm(complete_data - full_data) / data_norm
        complete_data[~np.isnan(missing_data_input)] = missing_data_input[~np.isnan(missing_data_input)]
        mod_acc = 1 - np.linalg.norm(complete_data - full_data) / data_norm

        if lr < stopping_lr:
            break
        iter += 1
    cluster_acc = 0
    if true_labels is not None:
        C = thrC(C, alpha1)
        y_x, CKSym_x = post_proC(C, K, d, alpha2)

        missrate_x = err_rate(true_labels, y_x)
        cluster_acc = 1 - missrate_x
    completion_acc = accuracy

    if cluster_model == "LRR":
        print("Low Rank Representation model")
    elif cluster_model == "SSC":
        print("Self-expressive model")
        
    print("Total Epochs %.1d ::" % (iter),
          "Missing Percentage: %.2f ::" % (100 * missing_data_perc),
          "Completion Accuracy: %.2f ::" % ((mod_acc) * 100),
          "Cluster Accuracy: %.2f " % (cluster_acc * 100))
    return [completion_acc, cluster_acc]

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    custom_model = ["CFS", "SSC"]
    print("Using device:", device)

    # Keeping K and d constant
    K = 4 # Set a constant K value
    d = 10   # Set a constant d value
    noise = 0
    m = 50
    n = 50

    print(f"Running model for d={d}, K={K}, noise={noise}, m={m}, n={n}")
    LRR_accuracy = [run_model(i/100,custom_model[0],m, n, d, K, noise) for i in range(0,100,10)]



### RUNNING MODEL on PRESET DATASETS

In [ ]:
import numpy as np
from DeLUCA import DeLUCA 
from custom_funcs import missing_data_generation, thrC, post_proC, err_rate, generate_data
from dataset_params import Dataset_params
from dataset_params import create_log
import torch
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=UserWarning)

def run_model(missing_data_perc,custom_model,dataset):
    torch.manual_seed(17)
    data, full_data, input_shape, batch_size, total_datapoints, \
    flat_layer_size, enc_layer_size, deco_layer_size, kernel_size, output_padding, \
    K, reg1, reg2, alpha1, alpha2, d, lr, rank, \
    true_labels = Dataset_params(dataset)   
    logs_path = create_log(dataset)
    cluster_model = custom_model  
    missing_data_input = missing_data_generation(data, int(missing_data_perc * total_datapoints))

    CAE = DeLUCA(input_shape, flat_layer_size, enc_layer_size, deco_layer_size, 
                kernel_size, output_padding, lr, K, rank, reg_const1=reg1, reg_const2=reg2,
                batch_size=batch_size, model_path=None, logs_path=logs_path,
                cluster_model=cluster_model,device=device)
    CAE.to(device)  # Move model to GPU
    stopping_lr = lr/20
    data_norm = np.linalg.norm(full_data)
    iter=1
    accuracy = 0
    while iter:
        C, cost, complete_data, lr = CAE.finetune_fit(missing_data_input)
        accuracy = 1 - np.linalg.norm(complete_data - full_data) / data_norm
        complete_data[~np.isnan(missing_data_input)] = missing_data_input[~np.isnan(missing_data_input)]
        mod_acc = 1 - np.linalg.norm(complete_data - full_data) / data_norm

        # Print epoch details
        # print("epoch: %.1d ::" % (iter + 1),
        #       "cost: %.8f ::" % (cost / data_norm),
        #       "accuracy: %.2f " % ((accuracy) * 100),
        #       "replace change: %.2f " % ((mod_acc) * 100),
        #       "learning rate: %.8f " % (lr))

        # Check if it's time to perform clustering
        if lr < stopping_lr:
            break
        iter+=1
    cluster_acc = 0
    if true_labels is not None:
        C = thrC(C, alpha1)
        y_x, CKSym_x = post_proC(C, K, d, alpha2)

        # Compute missrate and cluster accuracy
        missrate_x = err_rate(true_labels, y_x)
        cluster_acc = 1 - missrate_x
    completion_acc = accuracy

    if cluster_model == "LRR":
        print("Low Rank Representation model")
    elif cluster_model == "SSC":
        print("Self-expressive model")
        
    print("Total Epochs %.1d ::" % (iter),
          "Missing Percentage: %.2f ::" % (100 * missing_data_perc),
          "Completion Accuracy: %.2f ::" % ((mod_acc) * 100),
          "Cluster Accuracy: %.2f " % (cluster_acc * 100))
    return [completion_acc,cluster_acc]


if __name__ == '__main__': 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # device = torch.device("cpu")
    dataset = ["synthetic", "COIL20", "EYaleB", "ORL", "BostonHousing", "PeriodChanger", "DSDD", "HeartDisease", "HARUS", "Flowers", "Oxford_Pet"]
    custom_model = ["CFS","SSC"]
    print("Using device:", device)
    run_model(0.8,custom_model[0],dataset[3])

### Running the model on Novel Data

In [ ]:
import numpy as np
from DeLUCA import DeLUCA 
from custom_funcs import thrC, post_proC, err_rate, generate_data
from dataset_params import Dataset_params
from dataset_params import create_log
import torch
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def run_model(custom_model,dataset):
    data, full_data, input_shape, batch_size, total_datapoints, \
    flat_layer_size, enc_layer_size, deco_layer_size, kernel_size, output_padding, \
    K, reg1, reg2, alpha1, alpha2, d, lr, rank, \
    true_labels = Dataset_params(dataset)   
    logs_path = create_log(dataset)
    cluster_model = custom_model  

    CAE = DeLUCA(input_shape, flat_layer_size, enc_layer_size, deco_layer_size, 
                kernel_size, output_padding, lr, K, rank, reg_const1=reg1, reg_const2=reg2,
                batch_size=batch_size, model_path=None, logs_path=logs_path,
                cluster_model=cluster_model,device=device)
    CAE.to(device)  # Move model to GPU
    stopping_lr = lr/20
    data_norm = np.linalg.norm(full_data)
    iter=1
    while iter:
        C, cost, complete_data, lr = CAE.finetune_fit(data)
        complete_data[~np.isnan(data)] = data[~np.isnan(data)]
        
        # Print epoch details
        print("epoch: %.1d ::" % (iter + 1),
              "cost: %.8f ::" % (cost / data_norm),
              "learning rate: %.8f " % (lr))

        # Check if it's time to perform clustering
        if lr < stopping_lr:
            break
        iter+=1
    cluster_acc = 0
    if true_labels is not None:
        C = thrC(C, alpha1)
        y_x, CKSym_x = post_proC(C, K, d, alpha2)

        # Compute missrate and cluster accuracy
        missrate_x = err_rate(true_labels, y_x)
        cluster_acc = 1 - missrate_x

    # save_data_to_mat(missing_data_input, complete_data, full_data, missing_data_perc, dataset, logs_path)

    if cluster_model == "LRR":
        print("Low Rank Representation model")
    elif cluster_model == "SSC":
        print("Self-expressive model")
        
    print("Total Epochs %.1d ::" % (iter))

    return complete_data, cluster_acc


if __name__ == '__main__': 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset = ["Novel_datset"]
    custom_model = ["CFS","SSC"]
    print("Using device:", device)

    # run it
    Complete_data, cluster_acc = run_model(custom_model[0], dataset[0])

    # 1) print out the clustering accuracy
    print(f"Cluster Accuracy: {cluster_acc*100:.2f}%")

    # 2) save the completed data to disk
    import scipy.io as sio
    sio.savemat(f"{dataset[0]}_imputed.mat", {'completed_data': Complete_data})
